In [2]:
import sys
from mlflow.tracking import MlflowClient

import numpy as np
import pandas as pd


from tqdm import tqdm

sys.path.append("../.")


pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 400)
pd.set_option("display.float_format", lambda x: "%.4f" % x)


%reload_ext autoreload
%autoreload 2

In [ ]:
# Initialize MLflow Client
client = MlflowClient(tracking_uri="http://potato.felk.cvut.cz:2222")

experiment_names = [
    "pelesjak_getml_xgboost",
    "pelesjak_lightgbm_aggregate_hyperparams",
    "pelesjak_lightgbm_hyperparams",
    "pelesjak_linear_dbformer_hyperparams",
    "pelesjak_linear_sage_hyperparams",
    "pelesjak_linear_tabular_aggregate_hyperparams",
    "pelesjak_linear_tabular_hyperparams",
    "pelesjak_resnet_sage_hyperparams",
    "pelesjak_resnet_tabular_aggregate_hyperparams",
    "pelesjak_resnet_tabular_hyperparams",
]

for experiment_name in experiment_names:
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        raise ValueError(f"Experiment '{experiment_name}' not found.")
    experiment_id = experiment.experiment_id

    # Get all runs for the experiment
    runs = client.search_runs(experiment_ids=[experiment_id], max_results=10000)

    runs_dict = []
    for r in tqdm(runs.to_list()):
        r_info = r.info.__dict__
        r_data = {k: v for d in r.data.to_dictionary().values() for k, v in d.items()}

        runs_dict.append({**r_info, **r_data})

    df = pd.DataFrame(runs_dict)

    df.to_csv(f"./data/{experiment_name}.csv")

100%|██████████| 175/175 [00:00<00:00, 373538.52it/s]


In [3]:
dataset_tasks_index = [
    ("ctu-ergastf1", "ergastf1-original"),
    ("ctu-expenditures", "expenditures-original"),
    ("ctu-geneea", "geneea-original"),
    ("ctu-geneea", "geneea-temporal"),
    ("ctu-hepatitis", "hepatitis-original"),
    ("ctu-imdb", "imdb-original"),
    ("ctu-mondial", "mondial-original"),
    ("ctu-movielens", "movielens-original"),
    ("ctu-musklarge", "musklarge-original"),
    ("ctu-musksmall", "musksmall-original"),
    ("ctu-mutagenesis", "mutagenesis-original"),
    ("ctu-ncaa", "ncaa-original"),
    ("ctu-studentloan", "studentloan-original"),
    ("rel-amazon", "item-churn"),
    ("rel-amazon", "user-churn"),
    ("rel-avito", "user-clicks"),
    ("rel-avito", "user-visits"),
    ("rel-f1", "driver-dnf"),
    ("rel-f1", "driver-top3"),
    ("rel-stack", "user-badge"),
    ("rel-stack", "user-engagement"),
    ("rel-trial", "study-outcome"),
    ("ctu-accidents", "accidents-original"),
    ("ctu-accidents", "accidents-temporal"),
    ("ctu-craftbeer", "craftbeer-original"),
    ("ctu-dallas", "dallas-original"),
    ("ctu-dallas", "dallas-temporal"),
    ("ctu-diabetes", "diabetes-original"),
    ("ctu-financial", "financial-original"),
    ("ctu-financial", "financial-temporal"),
    ("ctu-genes", "genes-original"),
    ("ctu-hockey", "hockey-original"),
    ("ctu-legalacts", "legalacts-original"),
    ("ctu-legalacts", "legalacts-temporal"),
    ("ctu-premiereleague", "premiereleague-original"),
    ("ctu-premiereleague", "premiereleague-temporal"),
    ("ctu-tpcd", "tpcd-original"),
    ("ctu-webkp", "webkp-original"),
]

### Get overall results


In [ ]:
def get_topn_results(df, n=5):
    df = df[
        [
            "dataset_name",
            "task_name",
            "val_roc_auc",
            "test_roc_auc",
            "val_macro_f1",
            "test_macro_f1",
        ]
    ]
    df = df[
        (
            (df.val_roc_auc >= 0)
            & (df.val_roc_auc <= 1)
            & (df.test_roc_auc >= 0)
            & (df.test_roc_auc <= 1)
        )
        | (
            (df.val_macro_f1 >= 0)
            & (df.val_macro_f1 <= 1)
            & (df.test_macro_f1 >= 0)
            & (df.test_macro_f1 <= 1)
        )
    ]
    df_topn = []
    for val_metric in ["val_macro_f1", "val_roc_auc"]:
        # Drop rows where the validation metric is missing
        df_filtered = df.dropna(subset=[val_metric])

        # For each (dataset_name, task_name), get top N rows by validation metric
        df_topn.append(
            df_filtered.sort_values(val_metric, ascending=False)
            .groupby(["dataset_name", "task_name"], group_keys=False)
            .head(n)
        )

    return (
        pd.concat(df_topn)
        .groupby(["dataset_name", "task_name"])
        .aggregate(["mean", "std"])
        .reindex(index=dataset_tasks_index)
    )

In [160]:
lightgbm_df = pd.read_csv("./data/pelesjak_lightgbm_hyperparams.csv")

prop_df = pd.read_csv("./data/pelesjak_getml_xgboost.csv")
prop_df["_start_time"] = pd.to_datetime(prop_df["_start_time"], unit="ms")
prop_df_new = prop_df[(prop_df["_start_time"] > pd.Timestamp("2025-10-08 13:30:00"))]
prop_df_old = prop_df[prop_df["task_name"].str.contains("original")]
prop_df = pd.concat([prop_df_new, prop_df_old])


tabular_df = pd.read_csv("./data/pelesjak_resnet_tabular_hyperparams.csv")

linear_sage_df = pd.read_csv("./data/pelesjak_linear_sage_hyperparams.csv")
resnet_sage_df = pd.read_csv("./data/pelesjak_resnet_sage_hyperparams.csv")
dbformer_df = pd.read_csv("./data/pelesjak_linear_dbformer_hyperparams.csv")

N = 1
lightgbm_df = get_topn_results(lightgbm_df, n=N)
prop_df = get_topn_results(prop_df, n=N)
tabular_df = get_topn_results(tabular_df, n=N)
linear_sage_df = get_topn_results(linear_sage_df, n=N)
resnet_sage_df = get_topn_results(resnet_sage_df, n=N)
dbformer_df = get_topn_results(dbformer_df, n=N)

In [ ]:
overall_df_mean = pd.concat(
    [
        lightgbm_df,
        prop_df,
        tabular_df,
        linear_sage_df,
        resnet_sage_df,
        dbformer_df,
    ],
    axis=1,
    keys=(
        "LightGBM",
        "GetML XGBoost",
        "Tabular ResNet",
        "Linear SAGE",
        "ResNet SAGE",
        "DBFormer",
    ),
)
overall_df_mean = overall_df_mean.reset_index().rename(
    {"dataset_name": "Dataset", "task_name": "Task"}, axis=1
)

overall_df_mean.set_index(["Dataset", "Task"], drop=True, inplace=True)

bin_overall = (
    overall_df_mean.swaplevel(0, 1, axis=1)[["val_roc_auc", "test_roc_auc"]]
    .dropna(how="all")
    .swaplevel(0, 2, axis=1)["mean"]
    .sort_index(axis=1, level=[0, 1], ascending=[True, False])
    .swaplevel(0, 1, axis=1)
    .rename(
        columns={
            "val_roc_auc": "val",
            "test_roc_auc": "test",
        }
    )
    .swaplevel(0, 1, axis=1)
)[
    [
        "LightGBM",
        "GetML XGBoost",
        "Tabular ResNet",
        "Linear SAGE",
        "ResNet SAGE",
        "DBFormer",
    ]
]

mlt_overall = (
    overall_df_mean.swaplevel(0, 1, axis=1)[["val_macro_f1", "test_macro_f1"]]
    .dropna(how="all")
    .swaplevel(0, 2, axis=1)["mean"]
    .sort_index(axis=1, level=[0, 1], ascending=[True, False])
    .swaplevel(0, 1, axis=1)
    .rename(
        columns={
            "val_macro_f1": "val",
            "test_macro_f1": "test",
        }
    )
    .swaplevel(0, 1, axis=1)
)[
    [
        "LightGBM",
        "GetML XGBoost",
        "Tabular ResNet",
        "Linear SAGE",
        "ResNet SAGE",
        "DBFormer",
    ]
]

overall_df_mean = pd.concat([bin_overall, mlt_overall], axis=0)

overall_df_mean = overall_df_mean.round(3)


ranks_total = (
    overall_df_mean.swaplevel(0, 1, axis=1)["val"]
    .dropna(how="all")
    .rank(ascending=False, axis=1, na_option="bottom", method="average")
)


ranks_merged = overall_df_mean.swaplevel(0, 1, axis=1)["val"].dropna(how="all")
ranks_merged["RDL"] = ranks_merged[["Linear SAGE", "ResNet SAGE", "DBFormer"]].max(axis=1)
ranks_merged = ranks_merged[["LightGBM", "GetML XGBoost", "Tabular ResNet", "RDL"]].rank(
    ascending=False, axis=1, na_option="bottom", method="average"
)
ranks_merged_vals = np.zeros(6)
ranks_merged_vals[:4] = ranks_merged.mean(axis=0).values
overall_df_mean = overall_df_mean.swaplevel(0, 1, axis=1)
overall_df_mean.loc[("Avg. Rank", ""), "val"] = ranks_total.mean(axis=0).values
overall_df_mean.loc[("Avg. Rank Merged", ""), "val"] = ranks_merged_vals
overall_df_mean = overall_df_mean.swaplevel(0, 1, axis=1)


def highlight_max_val_test(row):
    is_val = row.index.get_level_values(1) == "val"
    is_test = row.index.get_level_values(1) == "test"

    styles = pd.Series("", index=row.index)

    val_max = row[is_val].max()
    styles.loc[(row == val_max) & is_val] = "font-weight: bold;"

    test_max = row[is_test].max()
    styles.loc[(row == test_max) & is_test] = "font-weight: bold;"

    return styles


overall_df_mean = overall_df_mean.reset_index()
ctu_mask = overall_df_mean["Dataset"].str.contains("ctu")
overall_df_mean.loc[ctu_mask, "Task"] = (
    overall_df_mean.loc[ctu_mask, "Task"].str.split("-").str[1]
).values
overall_df_mean["Dataset"] = (
    overall_df_mean["Dataset"].str.replace("ctu-", "").str.replace("rel-", "")
)

overall_df_mean.set_index(["Dataset", "Task"], drop=True, inplace=True)
overall_df_mean.style.apply(highlight_max_val_test, axis=1).to_latex(
    "./data/tables/overall_results.tex",
    convert_css=True,
    column_format="llcccccccccccc",
)